In [ ]:
%run ../common/libraries.py

# 이번 학습에서 사용하는 주요 라이브러리
# - pandas(pd) : CSV 및 DataFrame 데이터 처리
# - numpy(np) : 배열 및 수치 데이터 처리
# - train_test_split : 훈련/검증/테스트 데이터 분리
# - StandardScaler : 표준화 스케일링
# - PolynomialFeatures : 다항 특성 생성
# - KNeighborsRegressor : KNN 회귀 모델
# - LinearRegression : 선형/다중 회귀 모델
# - Ridge : L2 규제 회귀 모델
# - Lasso : L1 규제 회귀 모델
# - mean_absolute_error : MAE 평가
# - mean_squared_error : MSE 평가
# - r2_score : 결정계수(R²) 평가


In [ ]:
file_path = '../data/perch_dimensions.csv'
df  = pd.read_csv(file_path)

### 농어 무게
perch_weight = np.array(
    [5.9, 32.0, 40.0, 51.5, 70.0, 100.0, 78.0, 80.0, 85.0, 85.0,
     110.0, 115.0, 125.0, 130.0, 120.0, 120.0, 130.0, 135.0, 110.0,
     130.0, 150.0, 145.0, 150.0, 170.0, 225.0, 145.0, 188.0, 180.0,
     197.0, 218.0, 300.0, 260.0, 265.0, 250.0, 250.0, 300.0, 320.0,
     514.0, 556.0, 840.0, 685.0, 700.0, 700.0, 690.0, 900.0, 650.0,
     820.0, 850.0, 900.0, 1015.0, 820.0, 1100.0, 1000.0, 1100.0,
     1000.0, 1000.0]
     )

perch_full = df.to_numpy()
perch_full

train_input, test_input, train_target, test_target = train_test_split(
    perch_full, perch_weight, test_size=0.3, random_state=42
)
print(train_input.shape, test_input.shape)

In [ ]:
"""
## 다중회귀모델은 많은 특성을 이용할 수 있는 모델로,
 - 좋은 성능이 나타났음에도 좀 더 특성을 늘려서 성능향상이 가능한지 확인


<특성을 생성하는 라이브러리>
  - 패키기 : sklearn.preprocessing
  - 사용모델(클래스) : PolynomialFeatures(), 실제 모델은 아님
                    : "변환기" 모델이라고 칭합니다.
  - 사용함수 : fit(), transform()
            -> 훈련데이터와 검증데이터, 테스트데이터를 모두 변환할 경우에는 아래 순서를 따름
                - 1. 훈련데이터로 fit(훈련독립변수) 수행
                    - 변환할 때 패턴 찾기
                - 2. transform(훈련 or 검증 or 테스트 독립변수) 함수로 훈련 / 검증 / 테스트 독립변수를 각각 수행(검증 및 데트스에 대해서는 fit을 수행하지 않음)
                    - 찾은 변환 패턴으로 데이터 변환하기
            -> 훈련데이터만을 이용하여 간단히 모델성능을 확인하고자 할때는 빠른 수행을 위해 fit_transform()으로 1번과 2번을 동시에 수행하기도 함
 
<특성 생성 순서>
 - 변환기 모델 생성 : PolynomialFeatures()
 - 변환 패턴 찾기 : fit(훈련 독립변수)
 - 찾은 패턴으로 변환하기 : transform(훈련/검증/테스트 독립변수)


<특성을 추가하는 데이터>
 - 훈련/테스트, 또는 훈련/검증/테스트 데이터에서 독립변수만 변환합니다.
 - 종속변수는 원본 유지 합니다.
"""


## 특성 증가 예시

In [ ]:
### 샘플 데이터(독립변수)
temp_data = [[2,3,4]]

### 1. 변환기 클래스(특성 생성하는 클래스) 생성하기
# - include_bias = False : bias는 절편을 의미하며 0으로 보고 진행함 (절편 생략 = False 지정)
poly = PolynomialFeatures(include_bias = False)

### 2. 패턴 찾기
# - 모델 훈련시에는 종속변수도 함께 넣어줬으나, 변환기 클래스에서는 사용 안함
# - 독립변수의 데이터 변환이 목적임
poly.fit(temp_data)

### 3. 찾은 패턴을 이용 -> 독립변수의 특성 변환 시키기(특성 생성)
temp_poly =  poly.transform(temp_data)

In [ ]:
### fit()이 찾아낸 패턴(공식) 추출하기
poly.get_feature_names_out()

In [ ]:
temp_data = [[2,3,4]]

# - degree=n -> 특성 증가할 차원 정의(보통 4제곱승 까지 사용)
poly = PolynomialFeatures(degree=3,include_bias = False)

poly.fit(temp_data)

temp_poly =  poly.transform(temp_data)
temp_poly

In [ ]:
# 실제 사용하는 훈련 및 테스트 데이터를 이용하여 특성 생성하기

### 특성 생성할 차원은 2로
# - 실제변수 : train_input, test_input
# - 생성 변수명 : train_poly, test_poly
poly = PolynomialFeatures(degree=4,include_bias = False)

poly.fit(train_input, test_input)

train_poly =  poly.transform(train_input)
test_poly =  poly.transform(test_input)

print(train_input.shape, test_input.shape)
print(train_poly.shape, test_poly.shape)

In [ ]:
# 모델 훈련시키기

### 선형회귀 (다중회귀모델) 훈련 시키기
# - 훈련 후 훈련 및 테스트 정확도(결정계수) 확인
# - 과적합여부 해석
# - 테스트 데이터로 예측하기
# - MAE, MSE 오차 확인하기

In [ ]:
lr = LinearRegression()
lr

lr.fit(train_poly, train_target)

train_score = lr.score(train_poly, train_target)
test_score = lr.score(test_poly, test_target)
print()
print('특성 2차원: ',train_score, test_score, train_score-test_score)

train_pred = lr.predict(train_poly)
test_pred = lr.predict(test_poly)
print('테스트 예측')
print(test_pred)

train_mae = mean_absolute_error(train_target, train_pred)
test_mae = mean_absolute_error(test_target, test_pred)

train_mse = mean_squared_error(train_target, train_pred)
test_mse = mean_squared_error(test_target, test_pred)

train_r2 = r2_score(train_target, train_pred)
test_r2 = r2_score(test_target, test_pred)

print(f"훈련   - 절대오차 : {train_mae}, 제곱오차 : {train_mse},정확도 : {train_r2}")
print(f"테스트 - 절대오차 : {test_mae}, 제곱오차 : {test_mse},정확도 : {test_r2}")


### (해석)
# - 특성을 증가하기 전과 후를 비교했을 때
# - 특성을 증가한 경우 훈련 및 테스트 정확도가 높으며,
# - 과적합에 대해서도 나타나지 않으며, 오차도 매우 낮아졌음
# - 특성 증가후 차원2일때와 3일때를 비교한 결과
# - 3일때의 훈련 정확도가 매우 높게 나타났으나,
# - 과적합의 경우 차원 2일때 매우 낮으며, 오차도 낮게 나타났음
# - 따라서, 농어의 길이, 두께, 너비를 이용하여 무게를 예측하는 모델로는
# - 특성의 차원2로 증가한 다중회귀모델이 가장 일반화되어,
#   매우 훌륭한 모델로 판단됨(현재까지..)


In [ ]:
# <성능 높이는 방법>
# - 데이터양 늘리기
# - 하이퍼파라메터 튜닝(선형회귀에서는 튜닝 대상 모델이 아님, 변수 없음)
# - 다른모델 사용
# - 복잡도(x값=특성=컬럼=퓨처) 늘리기
# - 정규화(표준화)하기

In [ ]:
# 농어의 길이, 두께, 너비를 이용하여 무게를 예측
# - 데이터의 단위가 1~1000자리까지 편차가 큼
# - 따라서 데이터 스케일링 작업을 진행해 봄(성능에 영향을 미치는지 확인)
train_poly

In [ ]:
#데이터 스케일링 처리하기

### 표준화(standardScaler)만 수행해 보기
# - 훈련 및 테스트 데이터 모두 스케일링 후 다중회귀모델로 재훈련 시켜서 성능 확인하기

In [ ]:

ss = StandardScaler()

# 훈련 데이터 기준으로 학습
ss.fit(train_poly)

# 표준화
train_scaled = ss.transform(train_poly)
test_scaled = ss.transform(test_poly)

knr = KNeighborsRegressor(n_neighbors=5)

knr.fit(train_scaled, train_target)


train_score = knr.score(train_scaled, train_target)
test_score = knr.score(test_scaled, test_target)

print('KNN 정확도 체크 : ', train_score, test_score, train_score - test_score)


# 예측
train_pred = knr.predict(train_scaled)
test_pred = knr.predict(test_scaled)

print('테스트 예측')
print(test_pred)


# 오차 확인

## 평균절대오차
train_mae = mean_absolute_error(train_target, train_pred)
test_mae = mean_absolute_error(test_target, test_pred)

## 평균제곱오차
train_mse = mean_squared_error(train_target, train_pred)
test_mse = mean_squared_error(test_target, test_pred)

## 결정계수
train_r2 = r2_score(train_target, train_pred)
test_r2 = r2_score(test_target, test_pred)


print(f"훈련   - 절대오차 : {train_mae}, 제곱오차 : {train_mse}, 정확도 : {train_r2}")
print(f"테스트 - 절대오차 : {test_mae}, 제곱오차 : {test_mse}, 정확도 : {test_r2}")


### (해석)
# - 기존 가장 좋은 시점(차원2)의 정확도, 과적합여부, 오차와 비교 했을 때
# - 매우 미세한 차이로 정규화를 하지 않았을 때의 모델 성능이 더 좋았기에
# - 차원 2일때의 모델로 선정하는 것이 타당함

In [ ]:
# <성능 높이는 방법>
# - 데이터양 늘리기

# - 하이퍼파라메터 튜닝(선형회귀에서는 튜닝 대상 모델이 아님, 변수 없음)
# - 다른모델 사용 -> 튜닝이 가능한 개선된 모델 사용해보기...
# * <하이퍼파라메터 튜닝 가능 모델>
#   - 릿지(Ridge), 라쏘(Lasso), 앙상블모델 전체
#   - 훈련의 성능(과적합)에 영향을 미치는 변수의 값을 이용해서 조정하는 모델들

# - 복잡도(x값=특성=컬럼=퓨처) 늘리기 - ok
# - 정규화(표준화)하기 - ok

### 하이퍼 파라메터가 적용된 다른 모델 사용하기(릿지, 라쏘)

In [ ]:
"""
<사용 모델>
 - 사용 패키지 : sklearn.linear_model
 - 사용 모델(클래스) : Ridge, Lasso
 - 훈련에 사용되는 함수 : 기존 모델 훈련과 동일
 - 사용되는 하이퍼파라메터 : 규제강도(복잡도, 집중력)라는 개념이 적용됨


<모델 훈련시 주의할점>
 - 사람은 모델 훈련에 개입하지 않습니다.
"""

"""
<릿지, 라쏘에 사용되는 하이퍼파라메터>
 - 규제 속성명(변수명) : alpha
 - 규제 값의 범위 : 0.001 ~ 100 상이의 값중에 1개 사용
 - 규제 기본값(디폴트값, 생략가능) : 1
 - 보통 사용되는 값 : 0.001, 0.01, 0.1, 1, 10, 100 (기준은 1을 기준으로 비교)


<규제 값의 성격>
 - alpha 값이 작을 수록 : 훈련 정확도는 낮아지고, 과적합에 도움을 줄 수 있음
 - alpha 값이 높을 수록 : 훈련 정확도는 높아지며, 과적합에 도움이 안될 가능성이 높음
"""


In [ ]:
# 릿지 선언 및 규제강도 10으로 설정
ridge = Ridge(alpha=10)

ridge.fit(train_scaled, train_target)

train_score = ridge.score(train_scaled, train_target)
test_score = ridge.score(test_scaled, test_target)
print('특성 2차원: ',train_score, test_score, train_score-test_score)

train_pred = ridge.predict(train_scaled)
test_pred = ridge.predict(test_scaled)


train_mae = mean_absolute_error(train_target, train_pred)
test_mae = mean_absolute_error(test_target, test_pred)

train_mse = mean_squared_error(train_target, train_pred)
test_mse = mean_squared_error(test_target, test_pred)

train_r2 = r2_score(train_target, train_pred)
test_r2 = r2_score(test_target, test_pred)

print(f"훈련   - 절대오차 : {train_mae}, 제곱오차 : {train_mse},정확도 : {train_r2}")
print(f"테스트 - 절대오차 : {test_mae}, 제곱오차 : {test_mse},정확도 : {test_r2}")



In [ ]:
lasso = Lasso(alpha=10)

lasso.fit(train_scaled, train_target)

train_score = lasso.score(train_scaled, train_target)
test_score = lasso.score(test_scaled, test_target)
print('특성 2차원: ',train_score, test_score, train_score-test_score)

train_pred = lasso.predict(train_scaled)
test_pred = lasso.predict(test_scaled)

train_mae = mean_absolute_error(train_target, train_pred)
train_mse = mean_squared_error(train_target,train_pred)
train_r2 = r2_score(train_target,train_pred)

test_mae = mean_absolute_error(test_target, test_pred)
test_mse = mean_squared_error(test_target, test_pred)
test_r2 = r2_score(test_target, test_pred)

print(f"훈련   - 절대오차 : {train_mae}, 제곱오차 : {train_mse},정확도 : {train_r2}")
print(f"테스트 - 절대오차 : {test_mae}, 제곱오차 : {test_mse},정확도 : {test_r2}")

# -  라쏘(Lasso) 모델은 과소적합이 발생하였음(과소적합 해소가 필요함)

In [ ]:
alphas=[0.001,0.01,0.1,1,5,10,50,100]
train_row = {
    "model": [],
    "alpha" : [],
    "train_accuracy": [],
    "test_accuracy": [],
    "overfitting": [],
    "train_mae": [],
    "train_mse": [],
    "train_r2": [],
    "test_mae": [],
    "test_mse": [],
    "test_r2": []
}


for lasso_alpha in alphas :
    lasso = Lasso(alpha=lasso_alpha)
    train_row["model"].append(lasso.__class__.__name__)
    train_row["alpha"].append(lasso_alpha)
    lasso.fit(train_scaled, train_target)

    train_score = lasso.score(train_scaled, train_target)
    test_score = lasso.score(test_scaled, test_target)
    train_row["train_accuracy"].append(train_score)
    train_row["test_accuracy"].append(test_score)
    train_row["overfitting"].append(train_score-test_score)

    train_pred = lasso.predict(train_scaled)
    test_pred = lasso.predict(test_scaled)

    train_mae = mean_absolute_error(train_target, train_pred)
    train_mse = mean_squared_error(train_target,train_pred)
    train_r2 = r2_score(train_target,train_pred)
    train_row["train_mae"].append(train_mae)
    train_row["train_mse"].append(train_mse)
    train_row["train_r2"].append(train_r2)

    test_mae = mean_absolute_error(test_target, test_pred)
    test_mse = mean_squared_error(test_target, test_pred)
    test_r2 = r2_score(test_target, test_pred)
    train_row["test_mae"].append(test_mae)
    train_row["test_mse"].append(test_mse)
    train_row["test_r2"].append(test_r2)

for ridge_alpha in alphas :
    ridge = Ridge(alpha=ridge_alpha)
    train_row["model"].append(ridge.__class__.__name__)
    train_row["alpha"].append(ridge_alpha)
    ridge.fit(train_scaled, train_target)

    train_score = ridge.score(train_scaled, train_target)
    test_score = ridge.score(test_scaled, test_target)
    train_row["train_accuracy"].append(train_score)
    train_row["test_accuracy"].append(test_score)
    train_row["overfitting"].append(train_score-test_score)

    train_pred = ridge.predict([])
    test_pred = ridge.predict(test_scaled)

    train_mae = mean_absolute_error(train_target, train_pred)
    train_mse = mean_squared_error(train_target,train_pred)
    train_r2 = r2_score(train_target,train_pred)
    train_row["train_mae"].append(train_mae)
    train_row["train_mse"].append(train_mse)
    train_row["train_r2"].append(train_r2)

    test_mae = mean_absolute_error(test_target, test_pred)
    test_mse = mean_squared_error(test_target, test_pred)
    test_r2 = r2_score(test_target, test_pred)
    train_row["test_mae"].append(test_mae)
    train_row["test_mse"].append(test_mse)
    train_row["test_r2"].append(test_r2)


df = pd.DataFrame(train_row)
df

In [ ]:
"""
<분석 과정>
 1. 데이터 수집
 2. 데이터 전처리(확인 후 이상 없으면 바로 진행)
   2.1. 데이터 가공
 3. 가공된 데이터 특성간의 상관관계 확인
    -> 선형관계 또는 수평관계의 산점도 확인
       --> 선형관계 분포 : 회귀분석 진행
       --> 수평관계 분포 : 분류분석 진행
    -> 상관계수 확인(상관성이 있는지 확인)
 4. 특성들의 유의미성(p-value) 검증(또는 상관관계 검증이라고 칭합니다.)
 5. 상관관계 검증에 따른 데이터 후처리(가공)
   5.1 독립변수와 종속변수로 가공
 6. 훈련하기
     - 훈련을 위한 데이터 전처리하기 : 데이터 분류하기
                                   : 훈련vs테스트
                                   : 훈련vs검증vs테스트
     - 훈련모델 생성
     - 훈련 시키기
     - 훈련 및 테스트(검증) 정확도(또는 결정계수) 확인
       (과대 또는 과소적합 여부 확인)
       -- 과대 또는 과소적합인 경우 튜닝 진행
          -- 튜닝(정규화, 특성늘리기 등등...전처리 수행)
     - 예측하기
     - 오차 확인하기(평가하기)
       -- 평균절대오차(MAE)
       -- 평균제곱오차(MSE)
       -- 결정계수(R2)
       --> 모델 성능은 "결정계수"가 높고, 오차가 낮은값을 선택
           (단, 전제조건은 일반화 되었다는 가정하에~)
           
 7. 훈련 시 사용된 특성들 중 훈련에 중요하게 사용된 특성들 확인
    --> 특성중요도라고 칭합니다.
 8. 중요한 특성들을 기준으로 5번부터 다시 수행해 보기
 9. 하이퍼파라메터 튜닝 (정확도-결정계수 성능 올리기)
    --> 튜닝 클래스 또는 함수 사용
 10. 각 분석 방법에 속한 전체 모델 적용
 11. 최종 모델 선정
    - 6번의 오차(MAE, MSE, R2)를 모델별로 비교하여 선정
 12. 선정된 모델로 최종 테스트데이터로 예측 및 평가하기
 13. 선정된 "모델 저장"하기
 14. 서비스하고자하는 외부 프로그램에서 저장된 모델 불러들여서 사용하면 됨...
     - 외부 서비스하는 프로그램에서는 예측에 사용될 독립변수만 받아들이고,
     - 예측만 수행하면 됩니다.
     - 절대로, 서비스하는 곳에서는 모델을 훈련하거나 평가하면 안됩니다.
     - 웹(Web) 서비스를 하려면, Back-End 영역에 모델을 복사해서 넣은 후 사용
"""
